# **Data Pipline**

#### clean metadata, extract audio features with librosa, produce the master parquet

In [23]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import librosa
import numpy as np
import faiss

# Paths
# Paths
raw_dir = Path("D:/music_data/raw")
metadata_dir = raw_dir / "fma_metadata"
processed_dir = Path("D:/music_data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
# Load our cleaned track list
df = pd.read_parquet(processed_dir / "tracks_clean.parquet")
print(f"Tracks to process: {len(df)}")

Tracks to process: 106574


In [24]:
# Paths
audio_dir = Path("D:/music_data/raw/fma_large")
features_dir = Path("D:/music_data/essentia_features")
processed_dir = Path("D:/music_data/processed")
features_dir.mkdir(parents=True, exist_ok=True)

# Load track list
df = pd.read_parquet(processed_dir / "tracks_clean.parquet")
print(f"Tracks to process: {len(df)}")

Tracks to process: 106574


In [25]:
#extracts tempo, spectral features, and chroma
def extract_features(audio_path):
    try:
        y, sr = librosa.load(audio_path, duration=30)

        #BPM (_ throwaway veriable)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        tempo = float(np.asarray(tempo).flatten()[0])

        #energy
        energy = float(np.mean(librosa.feature.rms(y=y)))

        #key
        chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
        key = int(np.argmax(chroma.mean(axis=1)))

        return {"bpm" : tempo, "energy": energy, "key": key, "status": "ok"}

    except Exception  as e:
        return {"status": "error", "error": str(e)}


# run on all tracks
errors = []
skipped = 0

for _,row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
    track_id = row['track_id']
    out_path = features_dir / f"{str(track_id).zfill(6)}.json"

    # Skip if already done
    if out_path.exists():
        skipped += 1
        continue

    folder = str(track_id).zfill(6)[:3]
    audio_path = audio_dir/ folder / f"{str(track_id).zfill(6)}.mp3"

    if not audio_path.exists():
        errors.append(track_id)
        continue

    features = extract_features(audio_path)

    with open(out_path, 'w') as f:
        json.dump(features, f)

print(f"\nDone")
print(f"Skipped (already done): {skipped}")
print(f"Error:  {len(errors)}")

    

Extracting features:   0%|          | 0/106574 [00:00<?, ?it/s]

C:\Users\nakay\Documents\seniorProjectDocker\venv\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
C:\Users\nakay\AppData\Local\Temp\ipykernel_38244\4063112115.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, duration=30)
C:\Users\nakay\Documents\seniorProjectDocker\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
C:\Users\nakay\Documents\seniorProjectDocker\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=927
  warnings.warn(
C:\Users\nakay\Documents\seniorProjectDocker\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=6


Done
Skipped (already done): 25000
Error:  0


## **merge metadata + features into master Parquet**

In [26]:
print(f"df length: {len(df)}")
print(f"Sample track_id: {df['track_id'].iloc[0]}")

# Check a json file exists
test_id = df['track_id'].iloc[1]
test_path = features_dir / f"{str(test_id).zfill(6)}.json"
print(f"Test path: {test_path}")
print(f"Exists: {test_path.exists()}")

# Try loading it
with open(test_path) as f:
    test_features = json.load(f)
print(f"Features: {test_features}")

df length: 106574
Sample track_id: 2
Test path: D:\music_data\essentia_features\000003.json
Exists: True
Features: {'bpm': 86.1328125, 'energy': 0.11790983378887177, 'key': 7, 'status': 'ok'}


In [27]:
#merge feautures with meta data
df = pd.read_parquet(processed_dir / "tracks_clean.parquet")

records = []

# _ throw away variable
for _, row in tqdm(df.iterrows(), total=len(df), desc="Merging feautures"):
    track_id =  row['track_id']
    json_path = features_dir / f"{str(track_id).zfill(6)}.json"

    if not json_path.exists():
        continue

    with open(json_path) as f:
        features = json.load(f)

    if features.get('status') != 'ok':
        continue

    records.append({
        'track_id': track_id,
        'title': row['title'],
        'artist': row['artist'],
        'genre':    row['genre'],
        'duration': row['duration'],
        'subset':   row['subset'],
        'bpm':      features['bpm'],
        'energy':   features['energy'],
        'key':      features['key']
    })

merged = pd.DataFrame(records)
print(f"Merged tracks; {len(merged)}")
print(merged.head())

        

Merging feautures:   0%|          | 0/106574 [00:00<?, ?it/s]

Merged tracks; 106407
   track_id            title      artist    genre  duration  subset  \
0         2             Food        AWOL  Hip-Hop       168   small   
1         3     Electric Ave        AWOL  Hip-Hop       237  medium   
2         5       This World        AWOL  Hip-Hop       206   small   
3        10          Freeway   Kurt Vile      Pop       161   small   
4        20  Spiritual Level  Nicky Cook      NaN       311   large   

          bpm    energy  key  
0  161.499023  0.145215    0  
1   86.132812  0.117910    7  
2   99.384014  0.148828    7  
3  112.347147  0.188172    6  
4  123.046875  0.191729   10  


In [28]:
# Save merged dataset
out_path = processed_dir / "ab_fma_v3.parquet"
merged.to_parquet(out_path, index=False)
print(f"Saved {len(merged)} tracks to {out_path}")

Saved 106407 tracks to D:\music_data\processed\ab_fma_v3.parquet


# Phase 3: Generate Embedding

#### replace VGGish 128-dim with CLAP 512 dim for richer sonic similarity
#### CLAP(contrastive language-Audio Pretraining) capture timbral texture, mood, and stylistic nuance a much finer resolution
##### generates 512 dimentional vectors for each track

In [29]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060
VRAM: 8.6 GB


In [30]:
# test to see if it running
from transformers import ClapModel, ClapProcessor

print("Loading CLAP model...")
model     = ClapModel.from_pretrained("laion/clap-htsat-fused")
processor = ClapProcessor.from_pretrained("laion/clap-htsat-fused")
model     = model.cuda()
print("CLAP model loaded on GPU")

Loading CLAP model...


Loading weights:   0%|          | 0/477 [00:00<?, ?it/s]

CLAP model loaded on GPU


In [31]:
# pipeline to generate embedding
embeddings_dir = Path("D:/music_data/embeddings")
embeddings_dir.mkdir(parents=True, exist_ok=True)

model.eval()

errors = []
skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Generating embeddings"):
    track_id = row['track_id']
    out_path = embeddings_dir / f"{str(track_id).zfill(6)}.npy"

    # Skip if already done
    if out_path.exists():
        skipped += 1
        continue

    folder = str(track_id).zfill(6)[:3]
    audio_path = audio_dir / folder / f"{str(track_id).zfill(6)}.mp3"

    if not audio_path.exists():
        errors.append(track_id)
        continue

    try:
        y, sr = librosa.load(audio_path, sr=48000, duration=30, mono=True)
        inputs = processor(audio=y, sampling_rate=48000, return_tensors="pt")
        inputs = {k: v.to("cuda") if hasattr(v, 'to') else v for k, v in inputs.items()}

        with torch.no_grad():
            output = model.get_audio_features(**inputs)

        embedding = output.pooler_output.cpu().numpy().flatten()
        np.save(out_path, embedding)

    except Exception as e:
        errors.append(track_id)

print(f"\nDone")
print(f"Skipped: {skipped}")
print(f"Errors:  {len(errors)}")

Generating embeddings:   0%|          | 0/106574 [00:00<?, ?it/s]

C:\Users\nakay\AppData\Local\Temp\ipykernel_38244\1246134042.py:27: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=48000, duration=30, mono=True)
C:\Users\nakay\Documents\seniorProjectDocker\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



Done
Skipped: 24985
Errors:  162


In [32]:
#check how much data we retained
npy_files = list(embeddings_dir.glob("*.npy"))
print(f"Total embeddings: {len(npy_files)}")
print(f"Bad tracks skipped: {len(errors)}")
print(f"Coverage: {len(npy_files)/106574*100:.2f}%")

Total embeddings: 106412
Bad tracks skipped: 162
Coverage: 99.85%


# Phase 4
### Build FAISS Index
#### switch from HNSW flat to IVF+PQ to handle 25K - 100k 512-dim vectors

In [33]:
#Build IVF+PQ Index

index_dir = Path("D:/music_data/index")
index_dir.mkdir(parents=True, exist_ok=True)

# load all embeddings into a matrix
npy_files = sorted(embeddings_dir.glob("*.npy"))
track_ids = []
vectors = []

for f in tqdm(npy_files, desc="Loading embeddings"):
    track_id = int(f.stem)
    vec = np.load(f)
    track_ids.append(track_id)
    vectors.append(vec)

vectors = np.stack(vectors).astype(np.float32)
print(f"Matrix shape: {vectors.shape}")
      


Loading embeddings:   0%|          | 0/106412 [00:00<?, ?it/s]

Matrix shape: (106412, 512)


In [34]:
# FAISS index

# Normalize vectors for cosine similiarity
faiss.normalize_L2(vectors)

#build IVF-PQ index
n_vectors = len(vectors)
dimension = 512
nlist = 256 # number of clusters
m = 32 # number of sub-vectors for PQ

quantizer = faiss.IndexFlatL2(dimension)
index = faiss.IndexIVFPQ(quantizer, dimension, nlist, m, 8)

#Traain the index
print("Training index...")
index.train(vectors)
print("Training done!")

print("Adding vectors...")
index.add(vectors)
print(f"Total vectors in index: {index.ntotal}")

#save index
index_path = index_dir / "faiss_ivfpqV3.index"
faiss.write_index(index, str(index_path))
print(f"index saved to {index_path}")

#save track ID map
id_map_path = index_dir /"track_id_map_V3.json"
with open(id_map_path, 'w') as f:
          json.dump(track_ids, f)
print(f"Track ID map saved to {id_map_path}")


Training index...
Training done!
Adding vectors...
Total vectors in index: 106412
index saved to D:\music_data\index\faiss_ivfpqV3.index
Track ID map saved to D:\music_data\index\track_id_map_V3.json


In [35]:
#sanity check
# Test the index - find 5 similar tracks to track 2
index.nprobe = 64  # search 64 clusters for better recall

query_id  = track_ids.index(2)  # find position of track 2 in our list
query_vec = vectors[query_id].reshape(1, -1)

distances, indices = index.search(query_vec, 6)  # 6 because first result is the song itself

print(f"Query track: {df[df['track_id']==2][['title','artist','genre']].values[0]}")
print(f"\nTop 5 similar tracks:")
for i, idx in enumerate(indices[0][1:]):
    tid = track_ids[idx]
    row = df[df['track_id'] == tid][['title', 'artist', 'genre']].values
    if len(row) > 0:
        print(f"  {i+1}. {row[0]} (distance: {distances[0][i+1]:.4f})")

Query track: ['Food' 'AWOL' 'Hip-Hop']

Top 5 similar tracks:
  1. ["Don't Go" 'K.I.R.K' nan] (distance: 0.1902)
  2. ['Get Mines' 'CM aka Creative' 'Hip-Hop'] (distance: 0.2044)
  3. ['Scusatoia' 'ZONA MC' 'Hip-Hop'] (distance: 0.2054)
  4. ['Back Home' 'K.I.R.K' nan] (distance: 0.2080)
  5. ['Main Event Co Starring Kandid McFly - Angelous' 'K. Sparks' 'Hip-Hop'] (distance: 0.2110)


# Phase 5 Load data into MySQL